# Notebook 01 — 소표본·희소집단에서의 안정성

> **장점 #1**: 베이지안(Bayesian) 추론은 사전분포(prior)를 통한 정규화(regularization)로
> 표본이 작거나 집단 크기가 불균형(sparse focal group)할 때에도 안정적인 추정을 제공합니다.

### 학습 목표 (Learning Objectives)

- 표본 크기와 집단 균형이 DIF 검출에 미치는 영향을 시뮬레이션으로 확인한다.
- **빈도주의 MH**와 **베이지안 1PL DIF**의 추정 안정성을 비교한다.
- **RMSE, bias, 표집 SD**의 차이를 이해하고 **편향-분산 분해(bias-variance decomposition)**를 직접 본다.
- **명목 수준의 coverage(nominal coverage)** 개념을 이해하고 신용구간을 빈도주의적으로 점검한다.
- 점추정 SD를 관찰하는 **세 가지 이유**(MH 비교, 베이지안 절차의 빈도주의 평가, RMSE 분해)를 안다.

### 학습 전제

Notebook 00의 §1~§13을 먼저 학습하셨다고 가정합니다.
특히 §10(편향 vs 영향), §11(첫 베이지안 적합), §12(MH 비교)의 결과를 이해하고 있어야 합니다.


## 1. 본 노트북에서 점추정 SD를 관찰하는 이유

베이지안 추론은 본래 **사후분포 전체**를 제공합니다.
사후 평균과 95% 신용구간이 자연스러운 보고 단위입니다.
그런데 본 노트북에서는 **시뮬레이션 반복 간 점추정치의 표준편차**(표집 SD, sampling SD)를 명시적으로 봅니다.
이유는 다음 **세 가지**입니다.

### 이유 1 — MH와의 직접 비교 (공통 척도 필요)

빈도주의 Mantel-Haenszel(MH) 절차는 본 자료의 구현에서 within-sample 신뢰구간을 산출하지 않습니다.
따라서 두 방법을 *공통 척도*로 비교하려면 양쪽 모두에 적용 가능한
**반복 시뮬레이션 기반 평가**(RMSE, 표집 SD)가 필요합니다.

### 이유 2 — 베이지안 절차의 빈도주의적 평가 (frequentist evaluation)

베이지안 절차도 **빈도주의 성질(frequentist properties)** 로 평가될 수 있고 평가되어야 합니다.

- 점추정의 평균이 진짜 값에 가까운가? → **bias**
- 자료마다 추정이 얼마나 흔들리는가? → **표집 SD**
- 95% 신용구간이 진짜를 95% 비율로 포함하는가? → **명목 수준의 coverage**

이것이 *"베이지안이지만 빈도주의적 보증도 갖는가"* 를 묻는 표준 점검이며, simulation study의 핵심 도구입니다.

### 이유 3 — RMSE의 편향-분산 분해 (Bias-Variance Decomposition)

통계학의 기본 항등식:

$$
\mathrm{RMSE}^2 = \mathrm{Bias}^2 + \mathrm{Variance}_{\text{across-rep}}
$$

여기서 분산이 곧 **표집 SD의 제곱**입니다. RMSE 한 숫자만 보면 "왜 그 값인지" 모릅니다.
**bias와 SD를 함께 봐야** 방법 차이의 원인이 드러납니다.

| 측정량 | 의미 | 본 노트북 컬럼명 |
|---|---|---|
| **Bias** | "평균적으로 진짜에서 얼마나 벗어나 있는가" (정확성) | `bias = mean_est − truth` |
| **표집 SD** | "추정이 자료마다 얼마나 흔들리는가" (안정성) | `sd` |
| **RMSE** | 둘을 합친 종합 오차 | `rmse` |

베이지안의 prior shrinkage는 **약간의 bias를 도입하는 대신 SD를 크게 줄여** 전체 RMSE를 낮춥니다.
이 메커니즘을 시각화하는 것이 본 노트북의 핵심입니다.

### 두 종류의 SD 구분 (중요)

베이지안 분석에는 **서로 다른 두 가지 SD**가 있습니다.

| 명칭 | 정의 | 측정 대상 |
|---|---|---|
| **사후 SD (posterior SD)** | 한 번의 적합 안에서 사후분포의 표준편차 | 이 자료를 본 후의 모수 불확실성 |
| **표집 SD (sampling SD)** | N_REPS회 반복으로 얻은 점추정치들의 표준편차 | 절차의 빈도주의적 안정성 |

본 노트북의 `sd` 컬럼은 **표집 SD**입니다.
**잘 보정된(well-calibrated)** 베이지안 절차에서는 두 SD가 거의 일치합니다 (캘리브레이션).
이 점검은 §10의 명목 coverage 분석과 본질적으로 같은 작업입니다.


## 2. 시뮬레이션 시나리오 설정

다음 4개 시나리오를 반복 시뮬레이션(Monte Carlo replication)하여 두 방법의 추정 안정성을 비교합니다.

| 시나리오 | n_ref | n_focal | 의도 |
|---|---|---|---|
| **A — 균형 충분** | 300 | 300 | 표준, 두 방법이 유사할 것으로 예상 |
| **B — 균형 소표본** | 80  | 80  | 표본 적음, 베이지안 우위 시작 |
| **C — 희소 focal** | 300 | 50  | 집단 불균형, 베이지안 우위 확대 |
| **D — 극단 희소** | 400 | 25  | 매우 극단적, MH 불안정성 극대화 |

자료생성과정(data-generating process, DGP)은 Notebook 00과 동일:
- 10문항, 문항 5에 $\Delta b = +0.8$ (강한 DIF), 문항 8에 $\Delta b = -0.4$ (약한 DIF).
- 두 집단의 능력 평균 동일 (impact 없음).

각 시나리오마다 **N_REPS = 30** 회 반복합니다 (학습 목적상 빠른 실행; 실제 연구는 100~1000회 권장).

**예상되는 결과**:
- A에서는 두 방법이 비슷한 RMSE.
- B → C → D로 갈수록 **MH의 RMSE가 베이지안보다 빠르게 악화**.
- 베이지안은 prior 정규화로 **분산이 작아져** RMSE가 잘 보존됨.
- 단, 베이지안에는 약간의 bias(0 쪽 shrinkage)가 도입됨 — 편향-분산 trade-off.


## 3. 백엔드 선택 및 모듈 import

Notebook 00과 동일한 방식. Windows에서 UTF-8 오류가 발생하면 Notebook 00 §5의 안내를 따르세요.


In [ ]:
# 백엔드 선택
BACKEND = "stan"
import platform, importlib.util, warnings
def _resolve(req):
    req = req.lower()
    if req == "numpyro":
        ok = (importlib.util.find_spec("jax") is not None
              and importlib.util.find_spec("numpyro") is not None)
        if not ok:
            warnings.warn("numpyro unavailable -> Stan fallback")
            return "stan"
    return "stan" if req != "numpyro" else "numpyro"
BACKEND = _resolve(BACKEND)
print(f"Active backend: {BACKEND}")


In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from difbayes import simulate, visualize, frequentist, diagnostics, models
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False, "font.size": 10})
print("Modules loaded.")


## 4. 빈도주의 MH 출력의 척도 — log-odds (로그-오즈) 정리

본격 시뮬레이션 전에 **두 방법의 추정량을 같은 척도로 표현**하는 변환을 정리합니다.
이 절은 **표준 용어**를 정확히 사용하여 혼란을 없애는 것이 목적입니다.

### 4.1. 정확한 용어

| 영어 표준 | 한국어 표준 | 정의 |
|---|---|---|
| **logit** = **log-odds** | **로짓** = **로그-오즈** | $\log[p/(1-p)]$ — 단일 확률의 로그-오즈 변환 |
| **log odds ratio (log OR)** | **로그-오즈비** | $\log(\alpha) = \log(\text{odds}_1 / \text{odds}_2)$ — 두 오즈의 비에 로그 |
| **log-odds scale** | **로그-오즈 스케일** | 위 두 양이 공통으로 존재하는 좌표축 |

**"logit-OR"는 비표준 축약**입니다. 정확히는 **로그-오즈비(log odds ratio)** 이며,
그 값이 존재하는 좌표축은 **로짓 스케일 = 로그-오즈 스케일**입니다.

### 4.2. 두 방법의 자연 출력 비교

| 방법 | 자연 출력 | 단위 |
|---|---|---|
| **MH** | 공통 승산비 $\alpha_{MH}$ 와 그 로그 $\log(\alpha_{MH})$ | **로그-오즈 스케일** |
| **Bayes** | 사후 평균 $\hat{\Delta b}$ | **로짓 = 로그-오즈 스케일** |

두 값은 *같은 단위(로그-오즈)*에 살고 있어 직접 비교 가능합니다.

### 4.3. Rasch 1PL과의 직접 연결

Rasch 1PL의 정의에서, 같은 능력 $\theta$ 의 응답자가 두 집단에 있을 때:

$$
\log\frac{\mathrm{odds}_{ref}}{\mathrm{odds}_{focal}} = (\theta - b_{ref}) - (\theta - b_{focal}) = b_{focal} - b_{ref} = \Delta b
$$

즉:

$$
\boxed{\;\log(\alpha_{MH}) \approx \Delta b\;}
$$

로그-오즈비는 두 로짓 값의 *차이*이므로 단위가 logit이 됩니다.
따라서 MH 출력을 $\log(\alpha_{MH})$ 로 두면 베이지안 사후 평균 $\Delta b$ 와 **직접 비교 가능**합니다.

### 4.4. ETS의 Delta_MH 와의 관계

`frequentist.py` 의 출력 중 `delta_mh` 는 다음 정의를 따릅니다 (ETS 관습).

$$
\Delta_{MH} = -2.35 \times \log(\alpha_{MH})
$$

`-2.35` 는 logit을 probit-like delta scale로 환산하기 위한 **근사 상수**일 뿐 정확한 등식이 아닙니다.
따라서 본 노트북에서는 ETS scale로 환산하지 않고 **로그-오즈 스케일을 유지**해 비교합니다.

다음 코드의 변환 한 줄이 이 결정을 구현합니다:

```python
log_odds_ratio = -m.delta_mh / 2.35    # equivalent to log(alpha_mh)
```


## 5. 시뮬레이션 모수 설정

진짜 모수와 반복 횟수를 정의합니다.

> 시간 안내: 4개 시나리오 x 30회 x 2개 방법 = 240회 적합.
> Stan 컴파일은 첫 1회만, 이후 캐시 사용. 총 5~15분 소요 예상.
> 시간이 부족하면 N_REPS = 10 으로 줄여 실행해보세요.


In [ ]:
N_REPS = 30   # 학습용 빠른 실행. 정밀 비교는 100~1000 권장.

SCENARIOS = [
    dict(name="A: balanced (300/300)", n_ref=300, n_focal=300),
    dict(name="B: small balanced (80/80)", n_ref=80,  n_focal=80),
    dict(name="C: sparse focal (300/50)", n_ref=300, n_focal=50),
    dict(name="D: extreme sparse (400/25)", n_ref=400, n_focal=25),
]

b_true = np.linspace(-2.0, 2.0, 10)
delta_b_true = np.zeros(10)
delta_b_true[4] = 0.8
delta_b_true[7] = -0.4
TRUE_J = len(b_true)
print(f"N_REPS = {N_REPS}, J = {TRUE_J}")
print(f"True Delta b: {delta_b_true.round(2)}")


## 6. 반복 시뮬레이션 실행

각 시나리오 x 반복마다:
1. 새 자료 생성 (seed 변경)
2. MH 적용 -> 점추정 (로그-오즈 스케일)
3. 베이지안 적합 -> 사후 평균 + 95% 신용구간

각 결과를 `results` 리스트에 dict 형태로 누적합니다.

> 주의 — MH 결과의 lo, hi 는 NaN: `frequentist.py` 가 현재 표준오차(Robins-Breslow-Greenland)를
> 산출하지 않기 때문입니다. 본 노트북의 핵심 비교(RMSE, 표집 SD)는 점추정만으로 가능하므로 영향 없습니다.


In [ ]:
results = []

for sc in SCENARIOS:
    print(f"\n=== Scenario {sc['name']} ===")
    for r in range(N_REPS):
        seed = 1000 * (1 + SCENARIOS.index(sc)) + r
        data = simulate.simulate_rasch_dif(
            n_ref=sc["n_ref"], n_focal=sc["n_focal"],
            b_true=b_true, delta_b_true=delta_b_true, seed=seed,
        )

        # MH on log-odds scale (matches Bayesian Delta b unit)
        # log(alpha_MH) = -delta_MH / 2.35
        mh = frequentist.mantel_haenszel_all(data.Y, data.group, n_strata=4)
        for j, m in enumerate(mh):
            log_odds_ratio = (-m.delta_mh / 2.35) if np.isfinite(m.delta_mh) else np.nan
            results.append(dict(
                scenario=sc["name"], rep=r, method="MH (log-odds)",
                item=j+1, est=log_odds_ratio, lo=np.nan, hi=np.nan,
                truth=delta_b_true[j],
            ))

        # Bayesian non-hierarchical
        fit = models.fit_rasch_dif(
            Y=data.Y, group=data.group, backend=BACKEND,
            n_chains=2, n_warmup=300, n_samples=500,
            prior_sigma_delta=1.0, seed=seed,
        )
        samples = fit["samples"]["delta"].reshape(-1, TRUE_J)
        for j in range(TRUE_J):
            s = samples[:, j]
            results.append(dict(
                scenario=sc["name"], rep=r, method="Bayes (weak prior)",
                item=j+1, est=s.mean(),
                lo=np.quantile(s, 0.025), hi=np.quantile(s, 0.975),
                truth=delta_b_true[j],
            ))
        if (r + 1) % 5 == 0:
            print(f"  rep {r+1}/{N_REPS}")

results_df = pd.DataFrame(results)
print(f"\nTotal rows: {len(results_df)}")
results_df.head()


## 7. 결과 집계 — RMSE, bias, 표집 SD

각 (시나리오, 방법, 문항) 그룹에 대해 N_REPS회의 점추정치들을 모아 다음을 계산합니다.

| 컬럼 | 정의 | 의미 |
|---|---|---|
| `mean_est` | 점추정치들의 평균 | 추정의 *중심* |
| `truth` | 진짜 모수 | 비교 기준 |
| `bias = mean_est - truth` | 평균 오차 | 정확성 |
| `sd` | 점추정치들의 표준편차 | **표집 SD**, 안정성 |
| `rmse` | sqrt(평균 제곱 오차) | 종합 오차 |

이론적으로 RMSE^2 = Bias^2 + SD^2 가 성립합니다 (편향-분산 분해).
표 마지막에 bias2_plus_sd2 = sqrt(bias^2 + sd^2) 를 추가하여 직접 점검합니다.


In [ ]:
agg = (results_df
       .assign(error=lambda d: d["est"] - d["truth"])
       .groupby(["scenario", "method", "item"])
       .agg(rmse=("error", lambda x: np.sqrt(np.nanmean(x**2))),
            sd=("est", "std"),
            mean_est=("est", "mean"),
            truth=("truth", "first"))
       .reset_index())
agg["bias"] = agg["mean_est"] - agg["truth"]
agg["bias2_plus_sd2"] = np.sqrt(agg["bias"]**2 + agg["sd"]**2)
print("First 8 rows:")
print(agg.head(8).round(3).to_string(index=False))
print()
print("Decomposition check: max |rmse - sqrt(bias^2 + sd^2)| =",
      np.abs(agg['rmse'] - agg['bias2_plus_sd2']).max().round(4))


**점검**: `bias2_plus_sd2` 와 `rmse` 가 거의 같아야 합니다 (편향-분산 분해의 수치적 확인).
미세 차이는 N_REPS=30 의 Monte Carlo 변동이 원인입니다.


## 8. 시각화 1 — RMSE 시나리오별 비교

진짜 DIF 문항(5번, 8번)에 대한 RMSE 변화를 시각화합니다.
시나리오 A -> D 로 갈수록 두 방법의 RMSE 격차가 어떻게 변하는지가 핵심입니다.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for k, item in enumerate([5, 8]):
    sub = agg[agg["item"] == item]
    scenarios = [s["name"] for s in SCENARIOS]
    x = np.arange(len(scenarios))
    width = 0.35
    mh   = sub[sub["method"] == "MH (log-odds)"].set_index("scenario").loc[scenarios, "rmse"]
    bay  = sub[sub["method"] == "Bayes (weak prior)"].set_index("scenario").loc[scenarios, "rmse"]
    axes[k].bar(x - width/2, mh, width, label="MH (log-odds)", color="#1f77b4")
    axes[k].bar(x + width/2, bay, width, label="Bayes (weak prior)", color="#2ca02c")
    axes[k].set_xticks(x)
    axes[k].set_xticklabels([s.split(":")[0] for s in scenarios])
    axes[k].set_title(f"Item {item}  (true Delta b = {delta_b_true[item-1]:+.2f})")
    axes[k].set_ylabel("RMSE")
    axes[k].legend(fontsize=9)
    axes[k].grid(axis="y", alpha=0.3)
fig.suptitle("RMSE of DIF estimates across scenarios", y=1.02, fontsize=12)
fig.tight_layout()
fig.savefig("../outputs/01_rmse_comparison.png", dpi=120, bbox_inches="tight")
plt.show()


**해석 (1) — RMSE**

- 시나리오 A에서 두 방법은 비슷한 RMSE.
- B, C, D로 갈수록 **MH의 RMSE가 베이지안보다 빠르게 악화**.
- 왜? 다음 시각화(편향-분산 분해)에서 원인을 분석합니다.


## 9. 시각화 2 — 편향-분산 분해 (Bias-Variance Decomposition)

RMSE는 한 숫자라 **무엇 때문에** 커지는지(bias인지, SD인지) 알려주지 못합니다.
문항 5에 대해 bias와 SD를 시나리오별로 나란히 보면 메커니즘이 드러납니다.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

sub = agg[agg["item"] == 5]
scenarios = [s["name"] for s in SCENARIOS]
x = np.arange(len(scenarios))
width = 0.35

mh_b  = sub[sub["method"] == "MH (log-odds)"].set_index("scenario").loc[scenarios, "bias"]
bay_b = sub[sub["method"] == "Bayes (weak prior)"].set_index("scenario").loc[scenarios, "bias"]
axes[0].bar(x - width/2, mh_b, width, label="MH", color="#1f77b4")
axes[0].bar(x + width/2, bay_b, width, label="Bayes", color="#2ca02c")
axes[0].axhline(0, color="gray", lw=0.5)
axes[0].set_xticks(x)
axes[0].set_xticklabels([s.split(":")[0] for s in scenarios])
axes[0].set_ylabel("Bias (mean_est - truth)")
axes[0].set_title("Bias of Delta b estimates (Item 5)")
axes[0].legend(fontsize=9)
axes[0].grid(axis="y", alpha=0.3)

mh_s  = sub[sub["method"] == "MH (log-odds)"].set_index("scenario").loc[scenarios, "sd"]
bay_s = sub[sub["method"] == "Bayes (weak prior)"].set_index("scenario").loc[scenarios, "sd"]
axes[1].bar(x - width/2, mh_s, width, label="MH", color="#1f77b4")
axes[1].bar(x + width/2, bay_s, width, label="Bayes", color="#2ca02c")
axes[1].set_xticks(x)
axes[1].set_xticklabels([s.split(":")[0] for s in scenarios])
axes[1].set_ylabel("Sampling SD (across reps)")
axes[1].set_title("Sampling SD of Delta b estimates (Item 5)")
axes[1].legend(fontsize=9)
axes[1].grid(axis="y", alpha=0.3)

fig.tight_layout()
fig.savefig("../outputs/01_bias_variance.png", dpi=120, bbox_inches="tight")
plt.show()


**해석 (2) — 편향-분산 트레이드오프**

- **Bias 그림**: 베이지안은 prior가 진짜 +0.8을 약간 0 쪽으로 끌어당겨 **음의 bias**가 보임.
  특히 표본이 적을수록 prior 영향이 커서 bias 절댓값이 커짐.
- **SD 그림**: MH의 SD가 시나리오 D로 갈수록 폭증. 베이지안은 prior 정규화로 SD가 잘 억제됨.
- **종합**: 베이지안은 **약간의 bias를 도입**(나쁨)하지만 **SD를 크게 줄여**(좋음) 전체 RMSE를 낮춤.
  이것이 **편향-분산 트레이드오프(bias-variance tradeoff)**의 교과서적 예시입니다.

> Tip: "베이지안이 무조건 우월하다"가 아니라 **"베이지안은 약간의 체계적 편향을 감수하고
> 분산을 크게 줄이는 다른 trade-off를 채택한다"** 가 정확한 진술입니다.


## 10. 신용구간 Coverage — "명목 수준(nominal level)"의 의미

신용구간을 시뮬레이션으로 점검하기 전에, **명목 수준의 coverage** 개념을 정확히 정리합니다.

### 10.1. 정의

- **명목 수준 (nominal level)**: 우리가 *선언한* 신뢰도. "95% 신용구간"이라 부르면 명목 수준은 0.95.
  - "명목(nominal)"은 라틴어 nomen(이름)에서 왔으며, **"이름붙인 값"**, **"선언된 값"**을 뜻합니다.
- **실제 coverage (actual coverage)**: 반복 시뮬레이션에서 신용구간이 진짜 값을 *실제로* 포함한 비율.
- **명목 수준의 coverage**: 선언한 수준과 실제 coverage가 **일치**하는 상태.
  - 예: 0.95 선언 -> 실제 0.95 포함.

### 10.2. 세 가지 가능한 상태

| 실제 coverage | 명칭 | 진단 |
|---|---|---|
| ~ 0.95 | **Nominal (보정됨)** | 사후 SD ~ 표집 SD, 캘리브레이션 양호 |
| > 0.95 | **Conservative (보수적)** | 사후 SD > 표집 SD, 구간이 너무 넓음 (underconfident) |
| < 0.95 | **Anti-conservative (반보수적)** | 사후 SD < 표집 SD, 구간이 너무 좁음 (overconfident) |

### 10.3. 왜 점검하는가

베이지안 신용구간은 본래 *"이 자료를 봤을 때 모수가 이 구간에 있을 사후확률이 0.95"* 라는 **사후확률 진술**입니다.
빈도주의적 "장기 빈도 보장"은 자동으로 따라오지 않습니다 — prior와 모형이 적절할 때만 성립합니다.
따라서 시뮬레이션으로 **명목 coverage가 달성되는지 점검**하면 베이지안 절차의 캘리브레이션을 검증할 수 있습니다.

명목 coverage를 달성하면, 베이지안 신용구간은 **두 가지 보증**을 동시에 갖습니다:
- (a) "이 자료를 본 후 모수가 구간에 있을 사후확률 0.95"
- (b) "이 절차를 반복 적용하면 0.95 비율로 진짜를 포함"

### 10.4. 캘리브레이션과의 연결

§1에서 본 사후 SD와 표집 SD의 관계가 다시 등장합니다:

> 사후 SD ~ 표집 SD  ==>  실제 coverage ~ 명목 수준

두 SD의 일치를 점검하는 것이 곧 명목 coverage를 점검하는 것과 **본질적으로 같은 작업**입니다.
이는 베이지안의 **빈도주의적 평가(이유 2)** 의 실질적 도구입니다.

### 10.5. 어원적 직관

"명목(nominal)"은 통계학에서 **"이름붙인 값 vs 실제로 측정된 값"**의 대비를 표시할 때 씁니다.

| 명목 값 | 실제 값 | 일치하는가? |
|---|---|---|
| 명목 직경 1인치 나사 | 실측 0.997인치 | 거의 일치 (보정됨) |
| 명목 5% 유의수준 검정 | 실제 type-I error rate | 모형이 옳다면 일치 |
| **명목 95% 신용구간** | **실제 coverage** | **시뮬레이션으로 점검 가능** |


## 11. 시뮬레이션 기반 Coverage 점검

각 (시나리오, 문항) 그룹에서 베이지안 95% 신용구간이 진짜 값을 포함한 비율을 계산합니다.
이상적으로 0.95에 가까워야 합니다 (명목 수준의 coverage).


In [ ]:
bay = results_df[results_df["method"] == "Bayes (weak prior)"].copy()
bay["covered"] = (bay["lo"] <= bay["truth"]) & (bay["truth"] <= bay["hi"])
coverage = bay.groupby(["scenario", "item"])["covered"].mean().reset_index()
print("Bayesian 95% credible interval coverage:")
cov_table = coverage.pivot(index="item", columns="scenario", values="covered").round(2)
print(cov_table)


In [ ]:
# Scenario-level mean coverage with nominal line
fig, ax = plt.subplots(figsize=(8, 4))
scenarios = [s["name"] for s in SCENARIOS]
mean_cov = [coverage[coverage["scenario"]==s]["covered"].mean() for s in scenarios]
bars = ax.bar([s.split(":")[0] for s in scenarios], mean_cov, color="#2ca02c", alpha=0.7)
ax.axhline(0.95, color="red", ls="--", label="Nominal level (0.95)")
ax.set_ylim(0, 1.05)
ax.set_ylabel("Mean coverage (across items)")
ax.set_title("Bayesian 95% CI Coverage vs Nominal Level")
ax.legend()
ax.grid(axis="y", alpha=0.3)
for bar, val in zip(bars, mean_cov):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.02, f"{val:.2f}",
            ha="center", fontsize=10)
fig.savefig("../outputs/01_coverage.png", dpi=120, bbox_inches="tight")
plt.show()


**해석 (3) — Coverage 점검**

- 대부분의 시나리오에서 coverage가 **0.95 근처에 머무는 것**은 베이지안 절차가
  **명목 수준의 coverage**를 달성한다는 증거입니다.
- 시나리오 D(극단 희소)에서는 표본이 너무 작아 N_REPS=30 으로는 Monte Carlo 변동이 커서
  coverage가 다소 흔들릴 수 있습니다.
- coverage 평균이 0.95에서 멀리 떨어지면 **prior 민감도 분석** 또는 **위계 모형**(Notebook 03)을 고려해야 합니다.

> 베이지안의 신용구간이 빈도주의 신뢰구간으로서도 명목 수준을 달성한다는 것은
> *"이 베이지안 절차는 두 가지 보증을 동시에 갖는다"* 는 뜻입니다.
> 즉 (a) 사후확률 0.95 진술, 그리고 (b) 반복 적용 시 0.95 비율로 진짜 포함.
> 잘 보정된 베이지안 절차에서 이 둘이 일치합니다.


## 12. 요약 (Summary)

### 12.1. 개념적 정리

1. **점추정 SD를 보는 세 가지 이유**:
   - (a) MH와의 공통 척도 비교
   - (b) 베이지안 절차의 **빈도주의적 평가**
   - (c) RMSE의 **편향-분산 분해**

2. **두 종류의 SD 구분**:
   - **사후 SD** (within-sample): 한 번 적합 안의 사후분포 폭.
   - **표집 SD** (across-replication): 자료를 새로 뽑을 때마다 점추정 변동.

3. **로그-오즈 척도의 표준 용어**:
   - **로짓 (logit) = 로그-오즈 (log-odds)** — 단일 확률의 log-odds 변환.
   - **로그-오즈비 (log odds ratio)** — 두 오즈의 비에 로그.
   - **로그-오즈 스케일** — 위 둘이 공통으로 사는 좌표축.
   - "logit-OR"는 비표준 축약 — 정확히 표기하려면 "log odds ratio" 사용.

4. **명목 수준의 coverage (nominal coverage)**:
   - "선언한 신뢰도와 실제 포함률의 일치".
   - 잘 보정된 베이지안 절차의 핵심 지표.
   - 사후 SD ~ 표집 SD 이면 자동으로 달성.

### 12.2. 시뮬레이션 관찰

5. **편향-분산 트레이드오프**:
   - 베이지안은 **약간의 bias**를 도입하고 **SD를 크게 줄여** RMSE 우위 달성.
   - 소표본·희소집단으로 갈수록 효과 극대화.

### 12.3. 핵심 메시지

> 베이지안의 우위는 *마법이 아니라* **편향-분산 trade-off의 다른 선택**입니다.
> Prior가 약간의 bias를 도입하지만 분산을 크게 줄여, **소표본·희소집단 상황에서 RMSE 측면에서 더 안정**합니다.
> 동시에 신용구간이 **명목 수준의 coverage**를 달성하면 빈도주의적 보증도 함께 갖춥니다.

### 12.4. 다음 노트북

- **Notebook 02** — 사후확률 기반 풍부한 의사결정 (확률 진술, ROPE).
- **Notebook 03** — 위계 사전으로 다중검정 문제 자동 완화 (shrinkage).
- **Notebook 04** — Sparsity prior로 anchor-free DIF 검출.

### 12.5. 더 깊이 학습하려면

- **Simulation-Based Calibration (SBC)** — Talts et al. (2018), 표준 캘리브레이션 점검법.
- **Posterior Predictive Checks** — 모형이 자료를 재현하는지 확인.
- **Prior Sensitivity Analysis** — prior 폭을 0.5, 1.0, 2.0 등으로 변경해 결과 안정성 점검.
- **Robins-Breslow-Greenland 분산 추정량** — MH 신뢰구간 추가하여 coverage 양방향 비교 가능.
